# Matrículas por rede ao longo do tempo (INEP 2011–2025)

**Pergunta:** durante a pandemia houve uma possível migração de alunos da rede **privada** para a **pública**?

Este notebook olha as **matrículas somadas** (não a % de escolas com variação brusca) e mostra:

1. matrícula total por rede e ano;
2. índice com base fixa (ex.: 2019 = 100);
3. variação anual (%) por rede;
4. número de escolas ativas e matrícula média por escola;
5. as mesmas curvas separadas por etapa (anos iniciais, finais, ensino médio);
6. *(opcional)* decomposição da variação de matrículas entre dois anos em escolas que continuaram, que entraram e que saíram.

**Atenção:** o Censo Escolar usado aqui é agregado por escola. Não dá para acompanhar um aluno de uma escola para outra.
Os resultados são **indícios compatíveis** com migração, não uma prova. Outras explicações possíveis: abandono escolar,
fechamento de escolas, queda de natalidade, escolas que mudam de rede ou de código.

**Como usar:** edite só a célula da **Etapa 1 (parâmetros)** e rode as células em ordem.

## Etapa 0 — Imports

In [ ]:
%pip install vl-convert-python

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()


def estilizar(chart):
    """Visual limpo e consistente para todos os gráficos."""
    return (chart
            .configure_view(stroke=None)
            .configure_axis(gridColor="#E5E7EB", domainColor="#9CA3AF", tickColor="#9CA3AF",
                            labelFontSize=12, titleFontSize=13, labelColor="#374151", titleColor="#111827")
            .configure_title(fontSize=16, subtitleFontSize=12, subtitleColor="#6B7280", anchor="start", color="#111827")
            .configure_legend(labelFontSize=12, titleFontSize=12, symbolSize=120))


def mostrar(chart, nome):
    """Mostra o gráfico e, se SALVAR_IMAGENS = True, salva em PASTA_IMAGENS (precisa de vl-convert-python)."""
    display(chart)
    if SALVAR_IMAGENS:
        try:
            PASTA_IMAGENS.mkdir(parents=True, exist_ok=True)
            sufixo = f"{DIMENSAO}_{ETAPA}".replace("+", "mais")
            for fmt in FORMATOS_IMAGEM:
                destino = PASTA_IMAGENS / f"{nome}_{sufixo}.{fmt}"
                if fmt == "png":
                    chart.save(str(destino), scale_factor=ESCALA_PNG)
                else:
                    chart.save(str(destino))
            print("   salvo em", PASTA_IMAGENS / f"{nome}_{sufixo}.*")
        except Exception as e:
            print("[aviso] não foi possível salvar a imagem:", e,
                  "\n        Instale com: python -m pip install vl-convert-python")


pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

try:
    display
except NameError:                       # rodando fora do Jupyter
    def display(x):
        print(x)

## Etapa 1 — PARÂMETROS (edite só aqui)

In [ ]:
# ============================================================
#  PARÂMETROS — TUDO QUE VOCÊ PRECISA EDITAR ESTÁ NESTA CÉLULA
# ============================================================

# ---------- 1) ARQUIVO ----------
# Procura a pasta "dados" subindo a partir da pasta do notebook.
RAIZ = Path.cwd()
while not (RAIZ / "dados").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent

CAMINHO_ARQUIVO = RAIZ / "dados" / "bd5.zip"   # .zip (com um .csv dentro) ou direto .csv
NOME_ARQUIVO_NO_ZIP = None    # se o zip tiver vários arquivos, escreva o nome. None = usa o primeiro .csv
CSV_SEP      = ";"
CSV_ENCODING = "utf-8"        # se der erro de acentos, tente "latin-1"

# ---------- 2) QUAL "NÚMERO DE MATRÍCULAS" ANALISAR ----------
ETAPA            = "TOTAL"    # "TOTAL" (AI+AF+EM), "AI", "AF", "EM" ou soma de duas: "AI+AF", "AF+EM", "AI+EM"
INCLUIR_INTEGRAL = True       # True = regular + integral | False = só regular

# ---------- 3) COMO AGRUPAR AS ESCOLAS (cada grupo vira uma linha nos gráficos) ----------
DIMENSAO = "REDE"             # "REDE"        -> Pública (Federal+Estadual+Municipal) × Privada
                              # "DEP_ADM"     -> Federal, Estadual, Municipal e Privada separadas
                              # "LOCALIZACAO" -> Urbana × Rural
                              # "REGIAO" ou "UF"

# ---------- 4) FILTROS (None = não filtra) ----------
FILTRO_REDE        = None     # ex.: ["Pública"] ou ["Privada"]
FILTRO_DEP_ADM     = None     # ex.: ["Privada", "Municipal"]  (com DIMENSAO = "DEP_ADM" compara só essas duas)
FILTRO_LOCALIZACAO = None     # ex.: ["Urbana"]
FILTRO_REGIAO      = None     # ex.: ["Sudeste", "Sul"]
FILTRO_UF          = None     # ex.: ["RJ", "SP"]

# ---------- 5) PERÍODO ----------
ANO_INICIO = 2011
ANO_FIM    = 2025
ANO_BASE_INDICE = 2019        # ano que vale 100 no índice (use um ano antes da pandemia)
ANOS_DESTAQUE   = [2020, 2021]   # anos sombreados nos gráficos de linha (pandemia)

# ---------- 6) ETAPAS LADO A LADO (seção 6) ----------
ETAPAS_COMPARAR = ["AI", "AF", "EM"]     # aceita também somas, ex.: ["AI", "AF", "AI+AF", "TOTAL"]

# ---------- 7) ESCOLA ATIVA ----------
ESCOLA_ATIVA_MIN = 1          # a escola conta como "ativa" no ano se tiver pelo menos este nº de matrículas

# ---------- 8) DECOMPOSIÇÃO (seção 7, opcional) ----------
DECOMPOR = True               # False = pula a seção 7
PARES_DECOMPOSICAO = [(2019, 2020), (2020, 2021), (2021, 2022)]   # pares (ano anterior, ano)

# ---------- 9) GRÁFICOS ----------
LARGURA_GRAFICO = 900
ALTURA_GRAFICO  = 420
ESQUEMA_CORES   = "set2"      # paleta do Altair (ex.: "set2", "category10", "tableau10")

# ---------- 10) SALVAR IMAGENS DOS GRÁFICOS (para o pôster) ----------
SALVAR_IMAGENS  = True      # True = salva cada gráfico em PASTA_IMAGENS (precisa: python -m pip install vl-convert-python)
FORMATOS_IMAGEM = ["png"]     # "png", "svg" e/ou "pdf" (svg/pdf ficam vetoriais, melhores para pôster)
ESCALA_PNG      = 3           # resolução do PNG: 1 = tela, 3 = ~300 dpi, bom para impressão
PASTA_IMAGENS   = RAIZ / "analise" / "resultados" / "imagens"

# ---------- 11) EXPORTAÇÃO DE TABELAS ----------
EXPORTAR_CSV = False          # True = salva as tabelas em PASTA_SAIDA
PASTA_SAIDA  = RAIZ / "analise" / "resultados" / "matriculas_por_rede"

## Etapa 2 — Carregar a base

Cada linha da base é uma **escola em um ano**. A matrícula da escola é a soma das colunas `QT_MAT_*` da etapa escolhida
(regular, e integral se `INCLUIR_INTEGRAL = True`).
`REDE` é criada a partir de `DEP_ADM`: Federal, Estadual e Municipal → **Pública**; Privada → **Privada**.

In [ ]:
ETAPAS_VALIDAS = ["AI", "AF", "EM"]
COLS_CADASTRO = ["DEP_ADM", "LOCALIZACAO", "REGIAO", "UF"]
TODAS_MAT = [f"QT_MAT_{e}_{t}" for e in ETAPAS_VALIDAS for t in ("REG", "INT")]


def etapas_da_selecao(etapa):
    """Interpreta ETAPA: "TOTAL", "AI", "AF", "EM" ou somas como "AI+AF" (qualquer ordem)."""
    if str(etapa).strip().upper() == "TOTAL":
        return list(ETAPAS_VALIDAS)
    partes = [p.strip().upper() for p in str(etapa).split("+")]
    if any(p not in ETAPAS_VALIDAS for p in partes) or len(set(partes)) != len(partes):
        raise ValueError(f"ETAPA inválida: {etapa!r}. Use 'TOTAL', 'AI', 'AF', 'EM' ou somas de duas "
                         f"etapas diferentes, como 'AI+AF', 'AF+EM' ou 'AI+EM'.")
    return [p for p in ETAPAS_VALIDAS if p in partes]


def colunas_matricula(etapa, integral):
    cols = []
    for e in etapas_da_selecao(etapa):
        cols.append(f"QT_MAT_{e}_REG")
        if integral:
            cols.append(f"QT_MAT_{e}_INT")
    return cols


def adicionar_rede(df):
    """REDE: Federal/Estadual/Municipal -> "Pública"; Privada -> "Privada"."""
    dep = df["DEP_ADM"].astype("string").str.strip().str.lower()
    rede = pd.Series(pd.NA, index=df.index, dtype="object")
    rede[dep.isin(["federal", "estadual", "municipal", "pública", "publica"]).fillna(False)] = "Pública"
    rede[dep.eq("privada").fillna(False)] = "Privada"
    df = df.copy()
    df["REDE"] = rede
    return df


def _limpar(c):
    return str(c).replace("\ufeff", "").replace("ï»¿", "").strip()


def carregar(caminho):
    """Lê só as colunas necessárias, de um .csv ou de um .zip."""
    caminho = Path(caminho)
    precisa = ["ANO", "CO_ESCOLA"] + COLS_CADASTRO + TODAS_MAT
    leitura = dict(sep=CSV_SEP, encoding=CSV_ENCODING, low_memory=False,
                   usecols=lambda c: _limpar(c) in precisa)
    if caminho.suffix.lower() == ".zip":
        with zipfile.ZipFile(caminho) as z:
            validos = [n for n in z.namelist() if n.lower().endswith((".csv", ".txt")) and not n.startswith("__MACOSX")]
            if not validos:
                raise FileNotFoundError("Nenhum .csv encontrado dentro do zip.")
            nome = NOME_ARQUIVO_NO_ZIP or validos[0]
            print("Lendo do zip:", nome)
            with z.open(nome) as f:
                df = pd.read_csv(f, **leitura)
    else:
        print("Lendo:", caminho)
        df = pd.read_csv(caminho, **leitura)
    df.columns = [_limpar(c) for c in df.columns]
    return df


def preparar(raw, etapa):
    """Uma linha por escola/ano: ANO, CO_ESCOLA, GRUPO, MATRICULAS (já com filtros e período)."""
    cols = colunas_matricula(etapa, INCLUIR_INTEGRAL)
    faltando = [c for c in ["ANO", "CO_ESCOLA"] + cols if c not in raw.columns]
    if faltando:
        raise KeyError(f"Colunas ausentes na base: {faltando}")
    d = raw.copy()
    d["MATRICULAS"] = d[cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

    if d.duplicated(["CO_ESCOLA", "ANO"]).any():          # escola repetida no mesmo ano -> soma
        outras = {c: "first" for c in d.columns if c not in ("CO_ESCOLA", "ANO", "MATRICULAS")}
        d = d.groupby(["CO_ESCOLA", "ANO"], as_index=False).agg({**outras, "MATRICULAS": "sum"})

    d = d[(d["ANO"] >= ANO_INICIO) & (d["ANO"] <= ANO_FIM)]
    for col, filtro in [("REDE", FILTRO_REDE), ("DEP_ADM", FILTRO_DEP_ADM), ("LOCALIZACAO", FILTRO_LOCALIZACAO),
                        ("REGIAO", FILTRO_REGIAO), ("UF", FILTRO_UF)]:
        if filtro is not None:
            d = d[d[col].isin(filtro)]
    if DIMENSAO not in d.columns:
        raise KeyError(f"DIMENSAO={DIMENSAO!r} não existe na base. Opções: REDE, DEP_ADM, LOCALIZACAO, REGIAO, UF.")
    d = d.assign(GRUPO=d[DIMENSAO].fillna("(vazio)").astype(str))
    return d[["ANO", "CO_ESCOLA", "GRUPO", "MATRICULAS"]].reset_index(drop=True)


raw = adicionar_rede(carregar(CAMINHO_ARQUIVO))
print("Base bruta:", raw.shape, "| anos:", int(raw["ANO"].min()), "a", int(raw["ANO"].max()))

base = preparar(raw, ETAPA)
ANOS = list(range(int(base["ANO"].min()), int(base["ANO"].max()) + 1))
print(f"Etapa: {ETAPA} | integral: {INCLUIR_INTEGRAL} | agrupado por: {DIMENSAO}")
print("Observações escola-ano usadas:", len(base), "| escolas distintas:", base["CO_ESCOLA"].nunique())
print("Grupos:", sorted(base["GRUPO"].unique()))

## Etapa 3 — Matrícula total por grupo e ano

Soma das matrículas de todas as escolas do grupo. Faixa cinza = `ANOS_DESTAQUE`.

In [ ]:
def com_faixa(chart_linhas, anos_destaque, altura):
    """Sombreia os anos de destaque atrás do gráfico principal."""
    if not anos_destaque:
        return chart_linhas
    faixa = (alt.Chart(pd.DataFrame({"ANO": anos_destaque}))
             .mark_bar(color="#9CA3AF", opacity=0.15, size=40)
             .encode(x=alt.X("ANO:O"), y=alt.value(0), y2=alt.value(altura)))
    return alt.layer(faixa, chart_linhas)


def grafico_linhas(longo, y, titulo, y_titulo, zero=False, formato=",.0f"):
    cor = alt.Color("GRUPO:N", title=DIMENSAO, scale=alt.Scale(scheme=ESQUEMA_CORES))
    linhas = alt.Chart(longo).mark_line(point=alt.OverlayMarkDef(size=70, filled=True), strokeWidth=2.5).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(f"{y}:Q", title=y_titulo, scale=alt.Scale(zero=zero)),
        color=cor,
        tooltip=["ANO:O", "GRUPO:N", alt.Tooltip(f"{y}:Q", format=formato)])
    if zero is False and y == "VALOR":
        pass
    graf = com_faixa(linhas, ANOS_DESTAQUE, ALTURA_GRAFICO)
    return estilizar(graf.properties(width=LARGURA_GRAFICO, height=ALTURA_GRAFICO, title=titulo))


def longo(tab, nome="VALOR"):
    t = tab.copy()
    t.columns.name = "GRUPO"
    return t.reset_index().melt(id_vars="ANO", var_name="GRUPO", value_name=nome).dropna()


def total_por_grupo(d):
    return d.groupby(["ANO", "GRUPO"])["MATRICULAS"].sum().unstack("GRUPO").reindex(ANOS)


tot = total_por_grupo(base)
print("Matrícula total por grupo e ano")
display(tot.round(0).astype("Int64"))
mostrar(grafico_linhas(longo(tot), "VALOR", f"Matrícula total por {DIMENSAO} — etapa {ETAPA}", "Matrículas"), "01_matricula_total")

## Etapa 4 — Índice (ano base = 100) e variação anual

- **Índice:** matrícula do ano ÷ matrícula do `ANO_BASE_INDICE` × 100. Permite comparar redes de tamanhos muito diferentes na mesma escala.
- **Variação anual:** `(matrícula em t − matrícula em t−1) / matrícula em t−1`, em %.

In [ ]:
if ANO_BASE_INDICE not in tot.index or tot.loc[ANO_BASE_INDICE].isna().all():
    print(f"[aviso] ANO_BASE_INDICE={ANO_BASE_INDICE} não está na base filtrada; índice não calculado.")
    indice = None
else:
    indice = tot / tot.loc[ANO_BASE_INDICE] * 100
    print(f"Índice (ano {ANO_BASE_INDICE} = 100)")
    display(indice.round(1))
    g = grafico_linhas(longo(indice), "VALOR", f"Índice de matrículas por {DIMENSAO} ({ANO_BASE_INDICE} = 100) — etapa {ETAPA}",
                       f"Índice ({ANO_BASE_INDICE} = 100)", formato=".1f")
    mostrar(g, "02_indice")

var = tot.pct_change(fill_method=None) * 100        # só compara anos consecutivos
print("Variação anual das matrículas (%)")
display(var.round(2))

barras = (alt.Chart(longo(var)).mark_bar().encode(
    x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset("GRUPO:N"),
    y=alt.Y("VALOR:Q", title="Variação anual (%)"),
    color=alt.Color("GRUPO:N", title=DIMENSAO, scale=alt.Scale(scheme=ESQUEMA_CORES)),
    tooltip=["ANO:O", "GRUPO:N", alt.Tooltip("VALOR:Q", format=".2f", title="variação %")]))
zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(color="#374151").encode(y="y:Q")
mostrar(estilizar(alt.layer(barras, zero).properties(
    width=LARGURA_GRAFICO, height=ALTURA_GRAFICO,
    title=f"Variação anual das matrículas por {DIMENSAO} — etapa {ETAPA}")), "03_variacao_anual")

## Etapa 5 — Escolas ativas e matrícula média por escola

Ajuda a separar **"a escola encolheu"** de **"a escola deixou de existir"**.
Uma escola é *ativa* no ano se tiver pelo menos `ESCOLA_ATIVA_MIN` matrículas.
Matrícula média = matrícula total ÷ escolas ativas.

In [ ]:
ativas = base[base["MATRICULAS"] >= ESCOLA_ATIVA_MIN]
n_ativas = ativas.groupby(["ANO", "GRUPO"])["CO_ESCOLA"].nunique().unstack("GRUPO").reindex(ANOS)
media = (ativas.groupby(["ANO", "GRUPO"])["MATRICULAS"].sum().unstack("GRUPO").reindex(ANOS) / n_ativas)

print("Escolas ativas por grupo e ano")
display(n_ativas.astype("Int64"))
mostrar(grafico_linhas(longo(n_ativas), "VALOR", f"Escolas ativas por {DIMENSAO}", "Escolas ativas"), "04_escolas_ativas")

print("Matrícula média por escola ativa")
display(media.round(1))
mostrar(grafico_linhas(longo(media), "VALOR", f"Matrícula média por escola ativa, por {DIMENSAO}", "Matrículas por escola", formato=".1f"), "05_matricula_media")

## Etapa 6 — Mesmas curvas por etapa (índice e variação anual)

Para cada etapa de `ETAPAS_COMPARAR`, refaz o índice (`ANO_BASE_INDICE` = 100) e a variação anual,
usando os mesmos filtros e o mesmo agrupamento. Útil para ver em qual etapa o efeito é maior.

In [ ]:
indices, variacoes = [], []
for et in ETAPAS_COMPARAR:
    try:
        d_et = preparar(raw, et)
    except (KeyError, ValueError) as e:
        print(f"[aviso] etapa {et} ignorada: {e}")
        continue
    t = total_por_grupo(d_et)
    if ANO_BASE_INDICE in t.index and not t.loc[ANO_BASE_INDICE].isna().all():
        i = longo(t / t.loc[ANO_BASE_INDICE] * 100); i["etapa"] = et; indices.append(i)
    v = longo(t.pct_change(fill_method=None) * 100); v["etapa"] = et; variacoes.append(v)

if indices:
    ind_et = pd.concat(indices)
    graf = (alt.Chart(ind_et).mark_line(point=alt.OverlayMarkDef(size=45, filled=True), strokeWidth=2.2).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("VALOR:Q", title=f"Índice ({ANO_BASE_INDICE} = 100)", scale=alt.Scale(zero=False)),
        color=alt.Color("GRUPO:N", title=DIMENSAO, scale=alt.Scale(scheme=ESQUEMA_CORES)),
        tooltip=["etapa:N", "ANO:O", "GRUPO:N", alt.Tooltip("VALOR:Q", format=".1f")])
        .properties(width=max(220, LARGURA_GRAFICO // max(len(indices), 1) - 40), height=ALTURA_GRAFICO)
        .facet(column=alt.Column("etapa:N", title=None, header=alt.Header(labelFontSize=14, labelFontWeight="bold"))))
    mostrar(estilizar(graf.properties(title=f"Índice de matrículas por etapa ({ANO_BASE_INDICE} = 100)")), "06_indice_por_etapa")
    print("Índice por etapa e grupo")
    display(ind_et.pivot_table(index="ANO", columns=["etapa", "GRUPO"], values="VALOR").round(1))

if variacoes:
    var_et = pd.concat(variacoes)
    print("Variação anual (%) por etapa e grupo")
    display(var_et.pivot_table(index="ANO", columns=["etapa", "GRUPO"], values="VALOR").round(2))

## Etapa 7 — Decomposição da variação de matrículas entre dois anos *(opcional)*

Para cada par de anos `(a, b)` e cada grupo, a **variação líquida** das matrículas (soma de `b` menos soma de `a`)
é dividida em partes, olhando o **código da escola**:

| Componente | O que é |
|---|---|
| **continuantes: crescimento / redução** | escolas ativas nos dois anos, no mesmo grupo, que ganharam / perderam matrículas |
| **entrou: escola nova ou sem matrícula antes** | escola ativa em `b` e que não estava ativa em nenhum grupo em `a` |
| **entrou: veio de outro grupo** | escola ativa em `b` neste grupo e que em `a` estava ativa em outro grupo (mudou de rede, por exemplo) |
| **saiu: encerrou ou sem matrícula depois** | escola ativa em `a` e que não está ativa em nenhum grupo em `b` |
| **saiu: foi para outro grupo** | escola ativa em `a` neste grupo e que em `b` está ativa em outro grupo |

A soma das partes é a variação líquida do grupo. É uma **decomposição contábil das matrículas por escola**,
**não** o seguimento de alunos. Escola que muda de código (reabertura com outro código) aparece como saída + entrada.

In [ ]:
COMPONENTES = ["continuantes: crescimento", "continuantes: redução",
               "entrou: escola nova ou sem matrícula antes", "entrou: veio de outro grupo",
               "saiu: encerrou ou sem matrícula depois", "saiu: foi para outro grupo"]
CORES_COMP = ["#2E7D32", "#C62828", "#1565C0", "#64B5F6", "#6D4C41", "#FFB74D"]


def decompor(d, ano_a, ano_b):
    ativos = d[d["MATRICULAS"] >= ESCOLA_ATIVA_MIN]
    A = ativos[ativos["ANO"] == ano_a].set_index("CO_ESCOLA")[["GRUPO", "MATRICULAS"]]
    B = ativos[ativos["ANO"] == ano_b].set_index("CO_ESCOLA")[["GRUPO", "MATRICULAS"]]
    j = A.join(B, how="outer", lsuffix="_a", rsuffix="_b")
    ma, mb = j["MATRICULAS_a"].fillna(0), j["MATRICULAS_b"].fillna(0)
    ga, gb = j["GRUPO_a"], j["GRUPO_b"]

    mesmo = ga.notna() & gb.notna() & (ga == gb)
    dif = mb - ma
    partes = [
        (ga[mesmo & (dif > 0)], dif[mesmo & (dif > 0)], COMPONENTES[0]),
        (ga[mesmo & (dif < 0)], dif[mesmo & (dif < 0)], COMPONENTES[1]),
        (gb[gb.notna() & ga.isna()], mb[gb.notna() & ga.isna()], COMPONENTES[2]),
        (gb[gb.notna() & ga.notna() & (ga != gb)], mb[gb.notna() & ga.notna() & (ga != gb)], COMPONENTES[3]),
        (ga[ga.notna() & gb.isna()], -ma[ga.notna() & gb.isna()], COMPONENTES[4]),
        (ga[ga.notna() & gb.notna() & (ga != gb)], -ma[ga.notna() & gb.notna() & (ga != gb)], COMPONENTES[5]),
    ]
    linhas = []
    for grupo, valor, nome in partes:
        t = pd.DataFrame({"GRUPO": grupo, "valor": valor}).groupby("GRUPO").agg(valor=("valor", "sum"), escolas=("valor", "size"))
        t["componente"] = nome
        linhas.append(t.reset_index())
    out = pd.concat(linhas, ignore_index=True)
    out["par"] = f"{ano_a} → {ano_b}"
    return out


if not DECOMPOR:
    print("DECOMPOR = False: seção pulada.")
else:
    partes_dec = []
    for a, b in PARES_DECOMPOSICAO:
        if a in ANOS and b in ANOS:
            partes_dec.append(decompor(base, a, b))
        else:
            print(f"[aviso] par {a}→{b} fora do período analisado; ignorado.")
    if partes_dec:
        dec = pd.concat(partes_dec, ignore_index=True)
        dec["componente"] = pd.Categorical(dec["componente"], categories=COMPONENTES, ordered=True)

        tab = dec.pivot_table(index=["par", "GRUPO"], columns="componente", values="valor", aggfunc="sum", observed=False).fillna(0)
        tab["variação líquida"] = tab.sum(axis=1)
        print("Decomposição (matrículas)")
        display(tab.round(0).astype("Int64"))

        # confere com os totais da etapa 3
        conf = []
        for a, b in PARES_DECOMPOSICAO:
            if a in ANOS and b in ANOS:
                for g_ in tot.columns:
                    conf.append({"par": f"{a} → {b}", "GRUPO": g_, "variação total (etapa 3)": tot.loc[b, g_] - tot.loc[a, g_]})
        conf = pd.DataFrame(conf).set_index(["par", "GRUPO"])
        conf["variação líquida (decomposição)"] = tab["variação líquida"]
        conf["diferença"] = conf["variação total (etapa 3)"] - conf["variação líquida (decomposição)"]
        print("Conferência: a diferença deve ser 0 quando ESCOLA_ATIVA_MIN = 1")
        display(conf.round(0).astype("Int64"))

        graf = (alt.Chart(dec).mark_bar().encode(
            x=alt.X("GRUPO:N", title=None, axis=alt.Axis(labelAngle=0)),
            y=alt.Y("valor:Q", title="Variação de matrículas"),
            color=alt.Color("componente:N", title="Componente",
                            scale=alt.Scale(domain=COMPONENTES, range=CORES_COMP)),
            order=alt.Order("componente:N"),
            tooltip=["par:N", "GRUPO:N", "componente:N", alt.Tooltip("valor:Q", format=",.0f"), alt.Tooltip("escolas:Q", title="escolas")])
            .properties(width=max(140, LARGURA_GRAFICO // max(len(PARES_DECOMPOSICAO), 1) - 80), height=ALTURA_GRAFICO)
            .facet(column=alt.Column("par:N", title=None, header=alt.Header(labelFontSize=14, labelFontWeight="bold"))))
        mostrar(estilizar(graf.properties(title=f"Decomposição da variação de matrículas por {DIMENSAO} — etapa {ETAPA}")), "07_decomposicao")

## Etapa 8 — Exportação (opcional)

In [ ]:
if EXPORTAR_CSV:
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    tot.to_csv(PASTA_SAIDA / "matricula_total.csv", encoding="utf-8-sig")
    var.to_csv(PASTA_SAIDA / "variacao_anual_pct.csv", encoding="utf-8-sig")
    n_ativas.to_csv(PASTA_SAIDA / "escolas_ativas.csv", encoding="utf-8-sig")
    if indice is not None:
        indice.to_csv(PASTA_SAIDA / "indice.csv", encoding="utf-8-sig")
    if DECOMPOR and "dec" in globals():
        tab.to_csv(PASTA_SAIDA / "decomposicao.csv", encoding="utf-8-sig")
    print("Tabelas salvas em", PASTA_SAIDA)
else:
    print("EXPORTAR_CSV = False: nada salvo.")